In [2]:
import pandas as pd
import numpy as np


train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [3]:
for df in [train, test]:
    columns = list(df.columns)
    columns.remove("Date")
    for col in columns:
        for i in range(len(df)):
            if df[col].iloc[i] == "T":
                df.loc[i, col] = 0.01
            elif isinstance(df[col].iloc[i], str):
                df.loc[i, col] = float(df.loc[i, col])

In [4]:
# Preprocess the data by incrementing through the testing set in sets of 15 days. The first 14 days are input.

x_list, y_list = [], []

for i in range(0, len(train), 15):
    x = train[i: i + 14].drop("Date", axis=1)
    y = train.iloc[i + 14]['Minimum Temperature degrees (F)']
    # Here we convert to a numpy array and flatten so each variable from each day is its own feature
    # Using the difference from the previous day might also be helpful here
    x_list.append(np.array(x).flatten())
    y_list.append(y)

In [5]:
# Train a simple linear regression model to predict the temperature the next day
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(x_list, y_list)

In [6]:
# Finally, test the model on the test data and output a submission file

submission = pd.DataFrame(columns=["ID", 'Minimum Temperature degrees (F)'])

for i in range(0, len(test), 14):
    x = test[i: i + 14].drop("Date", axis=1)
    y = model.predict(np.array(x).flatten().reshape(1, -1))[0]
    submission.loc[len(submission)] = [int(i / 14), y]

In [7]:
submission

,ID,Minimum Temperature degrees (F)
0,0.0,53.188480
1,1.0,48.215110
2,2.0,51.770647
3,3.0,68.823759
4,4.0,62.420647
5,5.0,52.323964
6,6.0,73.290202
7,7.0,70.406930
8,8.0,51.392416
9,9.0,65.018465


In [8]:
submission.to_csv("submission.csv", index=False)